# Prompts and resources

Tools perform work. **Resources** are read-only data. A **prompt** is a template the server stores and the client can request.

[`research_server.py`](./research_server.py) already defines both, next to the two tools. This notebook is that part of the file.

## Related notes

- [Direct tool calling](./mcp.ipynb)
- [MCP server](./mcp-server.ipynb)
- [MCP client](./mcp-client.ipynb)
- [Reference servers](./reference-servers.ipynb)
- [Subprocess](./subprocess.ipynb)


## Resources

The research server writes each search to `papers/<topic>/papers_info.json`. This folder already has `papers/computers` and `papers/prediction_powered_interference`.

A resource is like a GET endpoint. It returns data. It should not do heavy work or change state.

This server exposes two resources.

- `papers://folders` lists topic folders.
- `papers://{topic}` returns the papers saved for one topic. The client fills in `{topic}` at call time.

`@mcp.resource(uri)` registers the function. The URI identifies the resource.


In [ ]:
@mcp.resource("papers://folders")
def get_available_folders() -> str:
    """List all available topic folders in the papers directory."""
    folders = []
    if os.path.exists(PAPER_DIR):
        for topic_dir in os.listdir(PAPER_DIR):
            topic_path = os.path.join(PAPER_DIR, topic_dir)
            papers_file = os.path.join(topic_path, "papers_info.json")
            if os.path.isdir(topic_path) and os.path.exists(papers_file):
                folders.append(topic_dir)

    content = "# Available Topics\n\n"
    if folders:
        for folder in folders:
            content += f"- {folder}\n"
        content += "\nUse @<topic> to read the papers in a topic folder.\n"
    else:
        content += "No topics found.\n"
    return content


@mcp.resource("papers://{topic}")
def get_topic_papers(topic: str) -> str:
    """Get detailed information about papers on a specific topic.

    Args:
        topic: The research topic to retrieve papers for.
    """
    topic_dir = topic.lower().replace(" ", "_")
    papers_file = os.path.join(PAPER_DIR, topic_dir, "papers_info.json")

    if not os.path.exists(papers_file):
        return (
            f"# No papers found for topic: {topic}\n\n"
            "Try searching for papers on this topic first."
        )

    try:
        with open(papers_file, "r") as papers_handle:
            papers_data = json.load(papers_handle)
    except json.JSONDecodeError:
        return (
            f"# Error reading papers data for {topic}\n\n"
            "The papers data file is corrupted."
        )

    content = f"# Papers on {topic.replace('_', ' ').title()}\n\n"
    content += f"Total papers: {len(papers_data)}\n\n"
    for paper_id, paper_info in papers_data.items():
        content += f"## {paper_info['title']}\n"
        content += f"- **Paper ID**: {paper_id}\n"
        content += f"- **Authors**: {', '.join(paper_info['authors'])}\n"
        content += f"- **Published**: {paper_info['published']}\n"
        content += (
            f"- **PDF URL**: [{paper_info['pdf_url']}]({paper_info['pdf_url']})\n\n"
        )
        content += f"### Summary\n{paper_info['summary'][:500]}...\n\n"
        content += "---\n\n"
    return content


## Prompt template

`@mcp.prompt()` registers a prompt. MCP uses the function name as the prompt name. It takes the arguments from the function arguments and the description from the docstring.

The prompt in this server is `generate_search_prompt`.


In [ ]:
@mcp.prompt()
def generate_search_prompt(topic: str, num_papers: int = 5) -> str:
    """Generate a prompt to find and discuss academic papers on a topic."""
    return f"""Search for {num_papers} academic papers about '{topic}' using the search_papers tool. Follow these instructions:
    1. First, search for papers using search_papers(topic='{topic}', max_results={num_papers})
    2. For each paper found, extract and organize the following information:
       - Paper title
       - Authors
       - Publication date
       - Brief summary of the key findings
       - Main contributions or innovations
       - Methodologies used
       - Relevance to the topic '{topic}'

    3. Provide a comprehensive summary that includes:
       - Overview of the current state of research in '{topic}'
       - Common themes and trends across the papers
       - Key research gaps or areas for future investigation
       - Most impactful or influential papers in this area

    4. Organize your findings in a clear, structured format with headings and bullet points for easy readability.

    Please present both detailed information about each paper and a high-level synthesis of the research landscape in {topic}."""


## Commands a client can offer

[`mcp_chatbot.py`](./mcp_chatbot.py) discovers tools only. A client that also uses prompts and resources would add three methods: `list_prompts`, `execute_prompt`, and `get_resource`.

Useful commands:

- `/prompts` lists prompt names.
- `/prompt generate_search_prompt topic=history num_papers=2` fills the template and sends it to the model.
- `@folders` reads `papers://folders`.
- `@computers` reads `papers://computers`.

`connect_to_server` would call `list_resources` and `list_prompts` beside `list_tools`. `chat_loop` would check for `/` and `@` before it sends the line to the model.


## Try the server

The inspector can list the resources and the prompt without those extra client commands.

From `MCP/`:

```bash
npx @modelcontextprotocol/inspector uv run --with "mcp>=1.2,<2" --with arxiv research_server.py
```

Open `papers://folders` and `papers://computers`. Then call the prompt `generate_search_prompt`.

The tool chatbot is still the command in [reference servers](./reference-servers.ipynb).
